# SMRD - stage 3: Grey Wolf hyperparameter search (Colab)

Tunes the BiGCN arm with GWO **inside** the 5-fold protocol, so the test folds stay untouched:

* For each test fold, 36 candidate settings (6 wolves x 5 rounds + the starting pack; wolf 0 = the stage 2 defaults).
* Each candidate is scored by validation macro-F1 on two training runs that use only the other four folds.
* The best candidate is then trained and tested once on the usual fold, and compared tweet by tweet with the untuned BiGCN.

Uses the same Drive folder as the cross-validation notebook (the untuned results there are needed for the comparison).
Every finished training run is logged to Drive; re-running a cell continues where it stopped.
**Run the last cell before switching or disconnecting the runtime.**

In [ ]:
#@title 1. GPU check and dependencies
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Runtime -> Change runtime type -> GPU (T4 or L4)')
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')
%pip install -q 'transformers==5.22.1' 'tokenizers==0.23.3' 'safetensors==0.8.0'     'huggingface_hub==2.4.0' 'pandas==3.1.0' 'scikit-learn==1.9.1' 'pytest==9.2.0'

In [ ]:
#@title 2. Mount Drive, get the stage3-gwo branch
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'stage3-gwo'
CKPT_ROOT = '/content/drive/MyDrive/smrd/checkpoints/cv'
os.makedirs(CKPT_ROOT, exist_ok=True)

PROJECT_ROOT = pathlib.Path('/content/smrd-reproduction')
if not PROJECT_ROOT.exists():
    !git clone -b {BRANCH} https://github.com/Miyonk/smrd-reproduction.git {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
os.chdir(PROJECT_ROOT)
!git log --oneline -1

In [ ]:
#@title 3. Unit tests (CPU, ~2 min) - stop here if anything fails
!python -m pytest tests/test_gwo.py tests/test_tune.py tests/test_cv.py tests/test_propagation.py -q

## twitter16 (run this first)

365 short BiGCN training runs. The log prints the time per run and the minutes left for the current fold.

In [ ]:
#@title 4a. twitter16: GWO search + final fit, 5 folds
!python -m src.run_tune search --dataset twitter16 --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4b. twitter16: comparison (graph_gwo vs graph is the last line)
!python -m src.run_cv compare --dataset twitter16 --ckpt-root {CKPT_ROOT}

## twitter15

About twice the Twitter16 time.

In [ ]:
#@title 5a. twitter15: GWO search + final fit, 5 folds
!python -m src.run_tune search --dataset twitter15 --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5b. twitter15: comparison
!python -m src.run_cv compare --dataset twitter15 --ckpt-root {CKPT_ROOT}

## pheme (optional, long)

Several times the Twitter15 time. Check the per-run time from the cells above before starting it.

In [ ]:
#@title 6a. pheme: GWO search + final fit, 5 folds
!python -m src.run_tune search --dataset pheme --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6b. pheme: comparison
!python -m src.run_cv compare --dataset pheme --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 7. All comparisons again (saved to Drive)
import glob, shutil
!python -m src.run_cv compare --all-datasets --ckpt-root {CKPT_ROOT}
for p in glob.glob('results/cv/*.json'):
    shutil.copy(p, CKPT_ROOT)

In [ ]:
#@title 8. Flush Drive (run before switching/disconnecting the runtime)
drive.flush_and_unmount()
print('Drive flushed and unmounted')